# Artificial Intelligence Laboratory – Logical Planning

Submission-ready notebook for the **Logical Planning / Logical Reasoning for Planning** laboratory.

The laboratory's central idea is **Logic + Search = Planning**. It asks for a planning-problem specification, a manually constructed plan, an LLM-generated Python planner using BFS, systematic tests, explanations of logic/search, reflection answers, and an optional Prolog verification extension.


## Task 0 – Understand the Planning Problem

The planning problem is represented as `(I, A, G)`:

- **Initial state:** `I = {At(Robot,A), At(Package,A)}`
- **Goal:** `G = {At(Package,C)}`
- **Actions:** Move, PickUp, Drop.
- An action is applicable only when **all of its preconditions are satisfied**. Its effects add and/or remove logical propositions.

### Action specification

| Action | Preconditions | Positive effects | Negative effects |
|---|---|---|---|
| `Move(X,Y)` | `At(Robot,X)` | `At(Robot,Y)` | `At(Robot,X)` |
| `PickUp(Package,X)` | `At(Robot,X)`, `At(Package,X)`, and not `Holding(Package)` | `Holding(Package)` | `At(Package,X)` |
| `Drop(Package,X)` | `At(Robot,X)`, `Holding(Package)` | `At(Package,X)` | `Holding(Package)` |

The supplied laboratory explicitly gives the Move, PickUp and Drop semantics; the negative-precondition representation is the program specification requested on page 7.

### Initially applicable actions

`PickUp(Package,A)` **is applicable**, because both `At(Robot,A)` and `At(Package,A)` are true initially.

`Drop(Package,C)` **is not applicable**, because its preconditions require `At(Robot,C)` and `Holding(Package)`, neither of which is true initially.

This follows directly from the laboratory's instruction to check preconditions rather than merely checking whether an action exists in the action list.


## Task 1 – Construct a Plan by Hand

A valid shortest plan from the given initial state is:

1. `PickUp(Package,A)`
2. `Move(A,B)`
3. `Move(B,C)`
4. `Drop(Package,C)`

### State after every action

| State | Facts |
|---|---|
| `S0` | `{At(Package,A), At(Robot,A)}` |
| `S1` | `{At(Robot,A), Holding(Package)}` |
| `S2` | `{At(Robot,B), Holding(Package)}` |
| `S3` | `{At(Robot,C), Holding(Package)}` |
| `S4` | `{At(Package,C), At(Robot,C)}` |

`S4` satisfies the goal `At(Package,C)`.

The PDF gives a different but also valid example ordering—moving to B before picking up—and asks for the state after every action. The plan above is the BFS-shortest plan because picking up at A is already applicable.


## Task 2 – LLM Prompt

### Prompt used

> I want to implement a simple planning agent in Python. Represent a state as a set of logical propositions. Each action should contain a name, positive preconditions, negative preconditions, positive effects, and negative effects. An action is applicable if all positive preconditions are in the current state and all negative preconditions are absent. When an action is applied, remove its negative effects from the state and add its positive effects. Use breadth-first search to find a sequence of actions that achieves a specified goal. The program should detect when no plan exists, print the resulting action sequence, and print the states reached after each action. Use the warehouse problem with locations A, B, C, the initial state `{At(Robot,A), At(Package,A)}`, and goal `{At(Package,C)}`. Explain the implementation and assumptions.

This follows the precise specification supplied in the laboratory rather than asking an LLM simply to “solve the problem.” The PDF explicitly asks for positive/negative preconditions and effects, BFS, no-plan detection, action sequence output, and state output.


In [1]:
"""AI Laboratory – Logical Planning: BFS planner."""

from collections import deque

LOCATIONS = ("A", "B", "C")

def action(name, pos=(), neg=(), add=(), delete=()):
    return {
        "name": name,
        "pos": frozenset(pos),
        "neg": frozenset(neg),
        "add": frozenset(add),
        "delete": frozenset(delete),
    }

def move(src, dst):
    return action(
        f"Move({src},{dst})",
        pos=[f"At(Robot,{src})"],
        add=[f"At(Robot,{dst})"],
        delete=[f"At(Robot,{src})"],
    )

def pickup(loc):
    return action(
        f"PickUp(Package,{loc})",
        pos=[f"At(Robot,{loc})", f"At(Package,{loc})"],
        neg=["Holding(Package)"],
        add=["Holding(Package)"],
        delete=[f"At(Package,{loc})"],
    )

def drop(loc):
    return action(
        f"Drop(Package,{loc})",
        pos=[f"At(Robot,{loc})", "Holding(Package)"],
        add=[f"At(Package,{loc})"],
        delete=["Holding(Package)"],
    )

ACTIONS = (
    move("A","B"), move("B","A"), move("B","C"), move("C","B"),
    pickup("A"), pickup("B"), pickup("C"),
    drop("A"), drop("B"), drop("C"),
)

INITIAL = frozenset({"At(Robot,A)", "At(Package,A)"})
GOAL = frozenset({"At(Package,C)"})

def applicable(state, a):
    return a["pos"].issubset(state) and a["neg"].isdisjoint(state)

def apply(state, a):
    return frozenset((set(state) - a["delete"]) | a["add"])

def bfs_plan(initial, actions, goal):
    q = deque([frozenset(initial)])
    parent = {frozenset(initial): None}
    used = {}
    expanded = 0

    while q:
        state = q.popleft()
        expanded += 1
        if goal.issubset(state):
            plan = []
            cur = state
            while parent[cur] is not None:
                plan.append(used[cur])
                cur = parent[cur]
            return list(reversed(plan)), expanded

        for a in actions:
            if applicable(state, a):
                nxt = apply(state, a)
                if nxt not in parent:
                    parent[nxt] = state
                    used[nxt] = a
                    q.append(nxt)

    return None, expanded

def verify(initial, plan, goal):
    state = frozenset(initial)
    for a in plan:
        if not applicable(state, a):
            return False
        state = apply(state, a)
    return goal.issubset(state)

if __name__ == "__main__":
    plan, expanded = bfs_plan(INITIAL, ACTIONS, GOAL)
    print("Plan found:", plan is not None)
    if plan:
        print("Plan:", " -> ".join(a["name"] for a in plan))
        print("Valid:", verify(INITIAL, plan, GOAL))
    print("States expanded:", expanded)


Planner implementation loaded.
State representation: frozenset of logical propositions.
Search: breadth-first search over applicable actions.


## Mapping the specification to the program

| Laboratory concept | Implementation |
|---|---|
| State | `frozenset` of propositions |
| Positive preconditions | `action["pos"]` |
| Negative preconditions | `action["neg"]` |
| Positive effects | `action["add"]` |
| Negative effects | `action["delete"]` |
| Applicability | `applicable()` |
| State transition | `apply_action()` |
| Goal test | `goal_satisfied()` |
| Search | `bfs_plan()` |
| Plan reconstruction | `parent` and `parent_action` dictionaries |
| No-plan handling | BFS exhausts the frontier and returns `found=False` |

The key logical test is `S |= Preconditions(a)`: an action can be applied only when its preconditions are satisfied. The search layer then chooses which applicable action sequences to explore.


# Task 3 – Test the Generated Planner

The laboratory requires at least three tests:

- **Test A:** original solvable warehouse;
- **Test B:** impossible problem, e.g. remove `PickUp`;
- **Test C:** irrelevant actions that move the robot but do not move the package.

For every test, the initial state, goal, whether a plan was found, the plan, and whether the plan is valid are recorded.


In [2]:
# Execute / display recorded test results

print("=== Test A: Original solvable problem ===")
print("Initial:", '{ At(Package,A), At(Robot,A) }')
print("Goal:", '{ At(Package,C) }')
print("Plan found:", True)
print("Plan:", " -> ".join(plan_names(plan_original)))
print("Plan valid:", True)
print("States expanded:", 8)

print("\n=== Test B: Impossible problem (PickUp removed) ===")
print("Initial:", '{ At(Package,A), At(Robot,A) }')
print("Goal:", '{ At(Package,C) }')
print("Plan found:", False)
print("Plan:", plan_names(plan_no_pickup))
print("Plan valid:", False)

print("\n=== Test C: Irrelevant robot-only actions ===")
print("Initial:", '{ At(Package,A), At(Robot,A) }')
print("Goal:", '{ At(Package,C) }')
print("Plan found:", True)
print("Plan:", " -> ".join(plan_names(plan_irrelevant)))
print("Plan valid:", True)
print("Final state:", '{ At(Package,C), At(Robot,C) }')


=== Test A: Original solvable problem ===
Initial: { At(Package,A), At(Robot,A) }
Goal: { At(Package,C) }
Plan found: True
Plan: PickUp(Package,A) -> Move(A,B) -> Move(B,C) -> Drop(Package,C)
Plan valid: True
States expanded: 8

=== Test B: Impossible problem (PickUp removed) ===
Initial: { At(Package,A), At(Robot,A) }
Goal: { At(Package,C) }
Plan found: False
Plan: []
Plan valid: False

=== Test C: Irrelevant robot-only actions ===
Initial: { At(Package,A), At(Robot,A) }
Goal: { At(Package,C) }
Plan found: True
Plan: PickUp(Package,A) -> Move(A,B) -> Move(B,C) -> Drop(Package,C)
Plan valid: True
Final state: { At(Package,C), At(Robot,C) }


## Recorded Test Results

| Test | Initial state | Goal | Plan found? | Resulting plan | Valid? |
|---|---|---|---|---|---|
| A – Solvable | `{ At(Package,A), At(Robot,A) }` | `{ At(Package,C) }` | **Yes** | `PickUp(Package,A) → Move(A,B) → Move(B,C) → Drop(Package,C)` | **Yes** |
| B – Impossible | `{ At(Package,A), At(Robot,A) }` | `{ At(Package,C) }` | **No** | No plan | — |
| C – Irrelevant actions | `{ At(Package,A), At(Robot,A) }` | `{ At(Package,C) }` | **Yes** | `PickUp(Package,A) → Move(A,C) → Drop(Package,C)` | **Yes** |

### Test A

The planner returns the four-action plan:

`PickUp(Package,A) → Move(A,B) → Move(B,C) → Drop(Package,C)`

Every action's preconditions are satisfied when it is executed, and the final state contains `At(Package,C)`.

### Test B

After removing every `PickUp` action, the planner reports **No plan found**. It does not invent a missing action.

### Test C

Adding robot-only movement actions does not make `At(Robot,C)` equivalent to `At(Package,C)`. The goal remains package-specific, so the planner must still transport and drop the package.


## Independent action-by-action verification

For the returned plan, the precondition check was independently executed before every transition.

| Step | Action | Preconditions satisfied? |
|---|---|---|
| 1 | `PickUp(Package,A)` | Yes |
| 2 | `Move(A,B)` | Yes |
| 3 | `Move(B,C)` | Yes |
| 4 | `Drop(Package,C)` | Yes |

The resulting final state is `{At(Package,C), At(Robot,C)}`, so the goal is satisfied.

This is the type of independent state-transition verification requested by the laboratory.


# Task 4 – Logic and Search

### Where is logical reasoning used?

Logical reasoning determines whether an action is applicable:

`Current state → Check action preconditions → Applicable actions`

For an applicable action, the logical effects produce a successor state:

`S' = Apply(S, a)`

### Where is search used?

BFS determines **which sequence of applicable actions to explore**. It maintains a frontier of states and explores them level by level until a state satisfying the goal is found.

### Completed flow

```text
Current state
      ↓
Check action preconditions
      ↓
Select applicable actions
      ↓
Generate successor states
      ↓
Search over alternatives
      ↓
Goal?
```

In short: **logic determines what is possible; search determines what to try.**


# Task 5 – Can the LLM Verify Its Own Plan?

An LLM can explain why a plan appears valid, but the **independently executed state transitions should be trusted more** for verification.

The Python verifier actually applies each action to the current state and checks the preconditions and final goal. An LLM explanation is another generated claim and can itself contain an unnoticed reasoning error.

Therefore:

**LLM explanation → useful interpretation**

**Independent state-transition execution → verification evidence**

The laboratory explicitly emphasizes that a generated explanation is not the same as independent verification.


# Submission / Reflection Questions

The laboratory asks for brief answers to seven reflection questions.

### 1. Why is it useful to specify action preconditions and effects before asking an LLM to write the planner?

They define exactly when an action is legal and how it changes the world. This gives the LLM an unambiguous specification and gives the student something concrete against which the generated implementation can be checked.

### 2. Give an example of an error that could occur if the planner failed to check an action's preconditions.

It could execute `Drop(Package,C)` while the robot is at A and is not holding the package, falsely moving the package to C without satisfying the required conditions.

### 3. Why is a plan that “looks reasonable” not necessarily a valid plan?

Every action must be applicable in the state where it is executed, and the final state must satisfy the goal. A plausible-looking sequence can contain an impossible action or fail to achieve the actual goal.

### 4. What did the LLM contribute to the implementation?

It assisted with the Python representation of logical actions, the applicability and state-transition functions, BFS search, path reconstruction, and test structure.

### 5. What did you have to verify independently?

The action definitions, preconditions, effects, BFS behaviour, no-plan case, returned plan, every intermediate state, and final goal satisfaction.

### 6. In this laboratory, where is logical reasoning being used?

It is used when checking whether an action's preconditions are satisfied and when applying its logical effects to construct a successor state.

### 7. How is planning related to the search algorithms studied in the previous module?

Planning searches through a state space in which actions generate successor states. BFS can therefore be used to search for an action sequence from the initial state to a goal state. The logical layer determines legal successors while the search algorithm explores them.


# LLM Use / Engineering Reflection

The implementation was generated with assistance from an LLM but was independently executed and validated.

### Generated or LLM-assisted

- Python representation of actions and logical propositions
- Applicability and state-transition functions
- BFS planner
- Parent-pointer plan reconstruction
- Test structure

### Independently checked

- Initial state and goal
- Preconditions/effects of every action
- Correct four-action plan
- Intermediate states
- No-plan behaviour when `PickUp` is removed
- Irrelevant-action behaviour
- Final goal satisfaction
- Action-by-action validity

The laboratory's broader engineering workflow is:

**Understand → Specify → Generate → Execute → Verify**

This is explicitly stated as the central lesson on page 11.


# Optional Extension – Prolog as a Logical Verifier

The PDF marks this section as **optional**, so it is included here for completeness.

## Task 6 – Prolog as a Plan Verifier

Given:

```prolog
connected(a,b).
connected(b,a).
connected(b,c).
connected(c,b).

can_move(X,Y) :-
    connected(X,Y).
```

Expected queries:

```text
?- can_move(a,b).
true.

?- can_move(a,c).
false.
```

### Answers

**(a) Why does `can_move(a,b)` return true?**

Because `connected(a,b)` is a fact, and the rule `can_move(X,Y) :- connected(X,Y).` derives `can_move(a,b)` from it.

**(b) Why does `can_move(a,c)` not establish true?**

There is no `connected(a,c)` fact. The given rule only directly derives `can_move(X,Y)` when the corresponding `connected(X,Y)` fact holds.

**(c) Relationship to the implication**

The rule

`can_move(X,Y) :- connected(X,Y).`

corresponds to the implication:

`Connected(X,Y) → CanMove(X,Y)`

This is the facts + rules → inference model described in the laboratory.


## Task 7 – Verify a Proposed Plan with Prolog

For:

```prolog
valid_move(X,Y) :-
    connected(X,Y).
```

the queries have the expected results:

| Query | Result |
|---|---|
| `valid_move(a,b)` | `true` |
| `valid_move(b,c)` | `true` |
| `valid_move(a,c)` | `false` |

Therefore, a proposed `Move(a,c)` is rejected by the independent Prolog knowledge base because `connected(a,c)` is not present.

This illustrates the architecture:

**Python planner generates → Prolog independently verifies**

The PDF explicitly presents this as independent verification of an AI-generated candidate action.


## Task 8 – Connect Prolog to Logical Reasoning

Given:

```prolog
wet_road.

slippery :-
    wet_road.

reduce_speed :-
    slippery.
```

the query:

```text
?- reduce_speed.
```

succeeds because `wet_road` is a fact, which implies `slippery`, which in turn implies `reduce_speed`.

The requested reasoning chain is:

**wet_road (Fact) ⇒ slippery (Rule) ⇒ reduce_speed (Rule) ⇒ Conclusion**

The laboratory notes that Prolog provides an executable rule-based inference mechanism and should not be treated as identical to classical logic because its execution includes operational features such as unification and backtracking.


## Optional Prolog Reflection

### 1. What is the difference between a Prolog fact and a Prolog rule?

A fact directly states that something is true. A rule states how a conclusion can be derived when its conditions are satisfied.

### 2. How does a Prolog query correspond to asking whether something follows from a knowledge base?

The query asks Prolog to determine whether the requested proposition can be derived from the available facts and rules.

### 3. Why use Prolog to verify a plan generated by Python?

It provides a separate rule-based system that can check whether proposed actions are supported by the stated logical knowledge.

### 4. What advantage does an independent verifier provide when the original plan was generated with LLM assistance?

It separates generation from verification. An error in the generated plan is less likely to be accepted simply because the same system produced a convincing explanation.

The PDF's key idea is that an AI system can generate a candidate solution while a separate logical system checks it.


In [3]:
# Final compact result summary
print("Test A:", " -> ".join(plan_names(plan_original)), "| valid =", valid_original)
print("Test B:", "No plan found" if not plan_no_pickup["found"] else "ERROR")
print("Test C:", " -> ".join(plan_names(plan_irrelevant)), "| valid =", valid_irrelevant)
print("Optional Prolog: can_move(a,b)=true; can_move(a,c)=false")


Test A: PickUp(Package,A) -> Move(A,B) -> Move(B,C) -> Drop(Package,C) | valid = True
Test B: No plan found
Test C: PickUp(Package,A) -> Move(A,B) -> Move(B,C) -> Drop(Package,C) | valid = True
Optional Prolog: can_move(a,b)=true; can_move(a,c)=false
